# Loading model

Loading in model from huggingface.

In [ ]:
from transformers import CLIPModel, CLIPProcessor
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

"""Checks if a cuda (NVIDIA GPU) is available and falls back to CPU if not"""
device = "cuda" if torch.cuda.is_available() else "cpu"

# Loads in CLIP model
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
model.eval()

# Loads in CLIP processor
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Creating embeddings

Firstly we load in the dataset

In [ ]:
from pathlib import Path
from PIL import Image

# Change this to "dataset" if your folder is named dataset/
root_dir = Path("images")

images = []
image_paths = []
labels = []

valid_ext = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}

for class_dir in sorted(root_dir.iterdir()):
    if not class_dir.is_dir():
        continue

    for img_path in sorted(class_dir.iterdir()):
        if img_path.suffix.lower() not in valid_ext:
            continue

        with Image.open(img_path) as img:
            images.append(img.convert("RGB"))
        image_paths.append(str(img_path))
        labels.append(class_dir.name)

print(f"Loaded {len(images)} images from {len(set(labels))} classes.")
print(f"First path: {image_paths[0] if image_paths else 'No images found'}")

We now have the dataset loaded in, and begin the process of creating embeddings. Utilise GPU if available NVIDIA GPU is detected, and if CUDA pytorch version is installed

In [ ]:
import gc
import numpy as np


batch_size = 500

image_embeddings = np.empty(shape=(0,512), dtype=float)

result_arrays = np.array_split(images, len(images) // batch_size)
    
for batch in result_arrays:
        with torch.inference_mode():
            inputs = processor(images=batch, return_tensors="pt")
            pixel_values = inputs["pixel_values"].to(device)

            features = model.get_image_features(pixel_values).pooler_output

            embeddings = features.detach().cpu().float().numpy()
            image_embeddings = np.concatenate((image_embeddings, embeddings), axis=0)
            del batch
            gc.collect()
            torch.cuda.empty_cache()


print(image_embeddings.shape)

# plot

## Functions

Define functions for plotting and analysing embeddings.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

In [ ]:
import typing as Any

def cluster_embeddings(x: np.ndarray, n_clusters: int = 5) -> np.ndarray:
	"""Step 6: Cluster embeddings with K-Means."""
	kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
	return kmeans.fit_predict(x)


def reduce_for_plot(x: np.ndarray, method: str = "tsne") -> np.ndarray:
	"""Step 7: Reduce embedding dimensions to 2D for visualization."""
	if method.lower() == "pca":
		reducer = PCA(n_components=2, random_state=42)
		return reducer.fit_transform(x)

	# t-SNE works better with moderate perplexity for small datasets.
	perplexity = min(30, max(5, len(x) // 10))
	reducer = TSNE(n_components=2, random_state=42, init="pca", perplexity=perplexity)
	return reducer.fit_transform(x)

In [ ]:
def plot_clusters(x_2d: np.ndarray, labels: np.ndarray, title: str) -> None:
	"""Create a scatter plot for semantic clusters."""
	plt.figure(figsize=(10, 7))
	unique_labels = np.unique(labels)
	colors = plt.cm.tab10(np.linspace(0, 1, len(unique_labels)))

	for i, cid in enumerate(unique_labels):
		mask = labels == cid
		plt.scatter(x_2d[mask, 0], x_2d[mask, 1], s=60, color=colors[i], label=f"Cluster {int(cid)}")

	plt.title(title)
	plt.xlabel("Component 1")
	plt.ylabel("Component 2")
	plt.legend(loc="best")
	plt.tight_layout()
	plt.show()

In [ ]:
def analyze_clusters(cluster_labels, true_labels, n_clusters):
    summary = {}
    grouped = defaultdict(list)

    for pred, truth in zip(cluster_labels, true_labels):
        grouped[int(pred)].append(truth)

    for cid in range(n_clusters):
        classes = grouped[cid]
        if classes:
            counts = Counter(classes)
            top_class, top_count = counts.most_common(1)[0]
            purity = top_count / len(classes)
        else:
            counts = {}
            top_class = None
            purity = 0.0

        summary[cid] = {
            "size": len(classes),
            "top_class": top_class,
            "purity": purity,
            "distribution": dict(counts),
        }

    return summary


## plotting embeddigs

In [ ]:
number_of_clusters = 10

lower_bound = 0
upper_bound = 10000



step_size = 1

x = image_embeddings[lower_bound:upper_bound:step_size]


print(x.shape)
print(x.shape[0])

n_clusters = min(5, x.shape[0])
cluster_labels = cluster_embeddings(x, n_clusters=number_of_clusters)
print(f"Assigned clusters for {len(cluster_labels)} images.")

x_2d = reduce_for_plot(x, method="tsne")
plot_clusters(x_2d, cluster_labels, "Semantic Clustering of Images using CLIP")

# Step 8 - Analyze clusters
summary = analyze_clusters(cluster_labels, labels[lower_bound:upper_bound:step_size], n_clusters=number_of_clusters)
print("\nCluster analysis:")
for cid in sorted(summary):
	item = summary[cid]
	print(
		f"Cluster {cid}: size={item['size']}, top_class={item['top_class']}, purity={item['purity']:.2f}"
	)


print("\n\n")

for cid in sorted(summary):
    item = summary[cid]
    print(
        f"Cluster {cid} \n {item['distribution']} \n \n"
    )

# Step 9 - Evaluate quality
silhouette = silhouette_score(x, cluster_labels) if len(np.unique(cluster_labels)) > 1 else -1.0
print(f"\nSilhouette Score: {silhouette:.4f}")
